# SSIF_V3 模型訓練 Notebook（繁體中文）

本 Notebook 使用 Google Drive 上**已預先切分**的觀測資料：

- `Data_formodel/training` → 全部作為正式 `train`
- `Data_formodel/validation` → 再分成 `validation` / `calibration` / locked `test`（50% / 25% / 25%，0.5 震級分箱）

科學隔離原則：**validation 選最佳 epoch；calibration 選 alert threshold；test 僅在兩者固定後評估。**

## 使用前（一次性）

1. 開啟共用資料夾：[Data_formodel](https://drive.google.com/drive/folders/1dwb-PH09nMaI5F5ZufW6mUMI5wU35J69)
2. 將資料放在（或捷徑指向）`MyDrive/00_SSIF/Data_formodel`，內含 `training/` 與 `validation/`
3. 掛載後路徑應為 `/content/drive/MyDrive/00_SSIF/Data_formodel/{training,validation}`
4. 勿下載／改寫原始資料；本 Notebook 只以 symlink 建立本機 staging archive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. 安全同步 repository

In [ ]:
from pathlib import Path
import os, shutil, subprocess

REPO_ROOT = Path('/content/SSIF_V3')
REPO_URL = 'https://github.com/oceanicdayi/SSIF_V3.git'

def run_checked(command, cwd=None, capture=False):
    result = subprocess.run(command, cwd=cwd, text=True, capture_output=capture)
    if capture:
        if result.stdout: print(result.stdout, end='')
        if result.stderr: print(result.stderr, end='')
    if result.returncode:
        raise RuntimeError(f"exit {result.returncode}: " + " ".join(map(str, command)))
    return result

os.chdir('/content')
if (REPO_ROOT / '.git').is_dir():
    try:
        run_checked(['git', '-C', str(REPO_ROOT), 'fetch', '--prune', 'origin'])
        run_checked(['git', '-C', str(REPO_ROOT), 'reset', '--hard', 'origin/main'])
        run_checked(['git', '-C', str(REPO_ROOT), 'clean', '-fd'])
    except RuntimeError:
        os.chdir('/content')
        shutil.rmtree(REPO_ROOT, ignore_errors=True)
        run_checked(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_ROOT)])
else:
    os.chdir('/content')
    shutil.rmtree(REPO_ROOT, ignore_errors=True)
    run_checked(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_ROOT)])

REPO_SHA = run_checked(['git', '-C', str(REPO_ROOT), 'rev-parse', 'HEAD'], capture=True).stdout.strip()
print('Repository commit:', REPO_SHA)
run_checked(['python', '-m', 'pip', 'install', '-q', '-r', str(REPO_ROOT / 'requirements.txt')])

## 2. 路徑與訓練設定

In [ ]:
from datetime import datetime, timezone
import gc, hashlib, json, math, platform, random, shutil, sys
from collections import defaultdict
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display

DRIVE_DATA_URL = 'https://drive.google.com/drive/folders/1dwb-PH09nMaI5F5ZufW6mUMI5wU35J69'
DATASET_ROOT = Path('/content/drive/MyDrive/00_SSIF/Data_formodel')
TRAIN_DATA = DATASET_ROOT / 'training'
VALIDATION_DATA = DATASET_ROOT / 'validation'
STAGED_ARCHIVE = Path('/content/ssif_preassigned_archive')

EXPECTED_TRAIN_COUNT = 773
EXPECTED_VALIDATION_COUNT = 193
EXPECTED_STAGED_COUNT = EXPECTED_TRAIN_COUNT + EXPECTED_VALIDATION_COUNT

WORK_ROOT = Path('/content/drive/MyDrive/00_SSIF/SSIF_V3_workspace')
PREPARED_VERSION = 'preassigned_v20260803'
PREPARED_DIR = WORK_ROOT / 'prepared' / PREPARED_VERSION
SPLIT_MANIFEST = PREPARED_DIR / 'split_manifest.json'
QUICK_DIAG_DIR = PREPARED_DIR / 'quick_diagnostic'
QUICK_SPLIT_MANIFEST = QUICK_DIAG_DIR / 'split_manifest.json'
QUICK_MODEL_DIR = WORK_ROOT / 'models' / 'quick_EW10_preassigned'
FULL_MODEL_DIR = WORK_ROOT / 'models' / 'ssif_v3_seed_20260803'
REPORT_DIR = WORK_ROOT / 'reports' / 'training_seed_20260803'
EXTERNAL_DATA = WORK_ROOT / 'data' / 'external_evaluation_json'
EXTERNAL_OUTPUT_DIR = WORK_ROOT / 'inference' / 'external_seed_20260803'

WINDOWS = [10,15,20,25,30,35,40]
SEED = 20260803
LABEL_HORIZON = 120
MIN_VALID = 0.80
MIN_PRECISION = 0.90
BATCH_SIZE = 16
EVAL_BATCH_SIZE = 64
WORKERS = 4
# A100 上可並行訓練多個 EW（共用一次已載入的 archive）。若 CUDA OOM，改成 1。
PARALLEL_EW_JOBS = 2

RUN_DATA_VALIDATION = True
CREATE_SPLIT_IF_MISSING = True
REBUILD_SPLIT = False
RUN_QUICK_TRAIN = True
RUN_FULL_TRAIN = False
RUN_EXTERNAL_EVALUATION = False
OVERWRITE_QUICK_MODEL = True
OVERWRITE_FULL_MODEL = False

for p in [PREPARED_DIR, QUICK_MODEL_DIR.parent, REPORT_DIR, EXTERNAL_OUTPUT_DIR.parent]:
    p.mkdir(parents=True, exist_ok=True)

if not DATASET_ROOT.is_dir():
    raise FileNotFoundError(
        f'找不到 DATASET_ROOT：{DATASET_ROOT}\n'
        f'請確認資料位於 /content/drive/MyDrive/00_SSIF/Data_formodel：{DRIVE_DATA_URL}'
    )
assert TRAIN_DATA.is_dir(), f'找不到 TRAIN_DATA：{TRAIN_DATA}'
assert VALIDATION_DATA.is_dir(), f'找不到 VALIDATION_DATA：{VALIDATION_DATA}'

def list_top_level_json(folder: Path):
    return sorted(p for p in folder.iterdir() if p.is_file() and p.suffix == '.json')

TRAIN_FILES = list_top_level_json(TRAIN_DATA)
VAL_FILES = list_top_level_json(VALIDATION_DATA)
assert len(TRAIN_FILES) == EXPECTED_TRAIN_COUNT, (
    f'training JSON 數量不符：got {len(TRAIN_FILES)}, expected {EXPECTED_TRAIN_COUNT}'
)
assert len(VAL_FILES) == EXPECTED_VALIDATION_COUNT, (
    f'validation JSON 數量不符：got {len(VAL_FILES)}, expected {EXPECTED_VALIDATION_COUNT}'
)
print('DATASET_ROOT:', DATASET_ROOT)
print('training JSON:', len(TRAIN_FILES))
print('validation JSON:', len(VAL_FILES))
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('PyTorch:', torch.__version__)


## 3. 建立來源 inventory 並 staging 扁平 archive

將 `training/` 與 `validation/` 的頂層 `*.json` 以 **symlink** 集中到 `/content/ssif_preassigned_archive`（不複製、不改動 Drive 原始檔）。正式訓練與此 archive + 凍結的 preassigned manifest 綁定。

In [ ]:
def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def build_source_inventory(files, source_role: str):
    rows = []
    for fp in files:
        rows.append({
            'source_role': source_role,
            'basename': fp.name,
            'source_path': str(fp.resolve()),
            'sha256': sha256_file(fp),
            'nbytes': fp.stat().st_size,
        })
    return rows

train_inventory = build_source_inventory(TRAIN_FILES, 'training')
val_inventory = build_source_inventory(VAL_FILES, 'validation')
all_inventory = train_inventory + val_inventory

basenames = [r['basename'] for r in all_inventory]
assert len(basenames) == len(set(basenames)), 'training/validation 之間或各自內部有重複檔名'

def inventory_hash(rows):
    payload = [
        {k: r[k] for k in ('source_role', 'basename', 'sha256', 'nbytes')}
        for r in sorted(rows, key=lambda x: (x['source_role'], x['basename']))
    ]
    return hashlib.sha256(json.dumps(payload, ensure_ascii=False, sort_keys=True).encode('utf-8')).hexdigest()

TRAIN_INVENTORY_SHA256 = inventory_hash(train_inventory)
VAL_INVENTORY_SHA256 = inventory_hash(val_inventory)
COMBINED_INVENTORY_SHA256 = inventory_hash(all_inventory)

if STAGED_ARCHIVE.exists():
    shutil.rmtree(STAGED_ARCHIVE)
STAGED_ARCHIVE.mkdir(parents=True, exist_ok=True)

for row in all_inventory:
    target = STAGED_ARCHIVE / row['basename']
    source = Path(row['source_path'])
    target.symlink_to(source)
    resolved = target.resolve()
    assert resolved == source.resolve(), f'symlink escape: {target} -> {resolved}'
    assert target.is_symlink()

staged_files = list_top_level_json(STAGED_ARCHIVE)
assert len(staged_files) == EXPECTED_STAGED_COUNT
assert all(p.is_symlink() for p in staged_files)

inventory_dir = PREPARED_DIR / 'source_inventory'
inventory_dir.mkdir(parents=True, exist_ok=True)
(inventory_dir / 'training_inventory.json').write_text(
    json.dumps(train_inventory, ensure_ascii=False, indent=2), encoding='utf-8'
)
(inventory_dir / 'validation_inventory.json').write_text(
    json.dumps(val_inventory, ensure_ascii=False, indent=2), encoding='utf-8'
)
(inventory_dir / 'inventory_hashes.json').write_text(json.dumps({
    'drive_data_url': DRIVE_DATA_URL,
    'dataset_root': str(DATASET_ROOT),
    'training_inventory_sha256': TRAIN_INVENTORY_SHA256,
    'validation_inventory_sha256': VAL_INVENTORY_SHA256,
    'combined_inventory_sha256': COMBINED_INVENTORY_SHA256,
    'expected_counts': {
        'training': EXPECTED_TRAIN_COUNT,
        'validation': EXPECTED_VALIDATION_COUNT,
        'staged': EXPECTED_STAGED_COUNT,
    },
}, ensure_ascii=False, indent=2), encoding='utf-8')

SOURCE_ROLE_BY_BASENAME = {r['basename']: r['source_role'] for r in all_inventory}
print('Staged archive:', STAGED_ARCHIVE)
print('Staged JSON:', len(staged_files))
print('combined_inventory_sha256:', COMBINED_INVENTORY_SHA256)

## 4. 使用與訓練相同的 loader 驗證 archive

以 `ssif_core.load_station_records()` 驗證 staging archive；同時分別檢查 training / validation 來源。

In [ ]:
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

REPORT_DIR.mkdir(parents=True, exist_ok=True)
environment = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(timespec='seconds').replace('+00:00', 'Z'),
    'repository_commit': REPO_SHA,
    'drive_data_url': DRIVE_DATA_URL,
    'dataset_root': str(DATASET_ROOT),
    'train_data': str(TRAIN_DATA),
    'validation_data': str(VALIDATION_DATA),
    'staged_archive': str(STAGED_ARCHIVE),
    'training_inventory_sha256': TRAIN_INVENTORY_SHA256,
    'validation_inventory_sha256': VAL_INVENTORY_SHA256,
    'combined_inventory_sha256': COMBINED_INVENTORY_SHA256,
    'python': platform.python_version(),
    'torch': torch.__version__,
    'cuda': torch.version.cuda,
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    'seed': SEED,
    'prepared_version': PREPARED_VERSION,
}
(REPORT_DIR / 'environment.json').write_text(
    json.dumps(environment, ensure_ascii=False, indent=2), encoding='utf-8'
)

sys.path.insert(0, str(REPO_ROOT))
from ssif_core import load_station_records, validate_event_split
from prepare_ssif_dataset import (
    _fingerprint,
    _read_event_metadata,
    _event_audits,
    common_cohort_keys,
    validate_split,
    _write_csv,
)
from dataclasses import asdict

archive_stats = None
if RUN_DATA_VALIDATION:
    records, archive_stats = load_station_records(
        STAGED_ARCHIVE,
        min_full_valid_fraction=MIN_VALID,
        min_series_length=max(WINDOWS),
        label_horizon=LABEL_HORIZON,
        require_label_horizon=True,
    )
    assert archive_stats['n_files'] == EXPECTED_STAGED_COUNT
    assert archive_stats['n_events'] > 0
    assert archive_stats['n_records'] > 0
    train_records, train_stats = load_station_records(
        TRAIN_DATA,
        min_full_valid_fraction=MIN_VALID,
        min_series_length=max(WINDOWS),
        label_horizon=LABEL_HORIZON,
        require_label_horizon=True,
    )
    val_records, val_stats = load_station_records(
        VALIDATION_DATA,
        min_full_valid_fraction=MIN_VALID,
        min_series_length=max(WINDOWS),
        label_horizon=LABEL_HORIZON,
        require_label_horizon=True,
    )
    report = {
        'status': 'ok',
        'loader': 'ssif_core.load_station_records',
        'staged_archive': str(STAGED_ARCHIVE.resolve()),
        'n_json_files_staged': EXPECTED_STAGED_COUNT,
        'stats_staged': archive_stats,
        'stats_training_source': train_stats,
        'stats_validation_source': val_stats,
    }
    (REPORT_DIR / 'archive_validation.json').write_text(
        json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8'
    )
    display(pd.DataFrame([
        {'archive': 'staged', **{k: archive_stats[k] for k in ['n_files', 'n_events', 'n_records', 'skipped_files']}},
        {'archive': 'training', **{k: train_stats[k] for k in ['n_files', 'n_events', 'n_records', 'skipped_files']}},
        {'archive': 'validation', **{k: val_stats[k] for k in ['n_files', 'n_events', 'n_records', 'skipped_files']}},
    ]))
    del records, train_records, val_records
    gc.collect()
    print('PASS: staged + source archives are usable by the training loader')

## 5. 建立或載入凍結的 preassigned split

政策：

1. **training 來源**的模型可用事件 → 只能進正式 `train`
2. **validation 來源**事件 → 以 seed=`20260803`、0.5 震級分箱，最大餘數法切成 validation / calibration / test = 50% / 25% / 25%
3. Quick EW10 使用**僅 training 來源**的診斷用四切分（`quick_diagnostic/`），不接觸 formal validation pool
4. 既有 formal artifacts 與目前 inventory / policy 不符時**失敗退出**（請改 `PREPARED_VERSION`）；不會自動覆寫

In [ ]:
def magnitude_bin_0p5(mag):
    if mag is None or not (isinstance(mag, (int, float)) and math.isfinite(mag)):
        return 'unknown'
    lo = math.floor(float(mag) * 2.0) / 2.0
    hi = lo + 0.5
    return f'{lo:.1f}-{hi:.1f}'

def largest_remainder_three_way(n, ratios=(0.50, 0.25, 0.25), roles=('validation', 'calibration', 'test')):
    exact = [n * r for r in ratios]
    floors = [int(math.floor(x)) for x in exact]
    left = n - sum(floors)
    order = sorted(range(len(roles)), key=lambda i: (-(exact[i] - floors[i]), i))
    counts = floors[:]
    for j in range(left):
        counts[order[j]] += 1
    return {roles[i]: counts[i] for i in range(len(roles))}

def allocate_validation_pool(event_items, seed=SEED):
    # event_items: list[(event_id, magnitude)] from validation source only.
    roles = ('validation', 'calibration', 'test')
    ratios = (0.50, 0.25, 0.25)
    ratio_map = dict(zip(roles, ratios))
    rng = random.Random(seed)

    by_bin = defaultdict(list)
    for eid, mag in event_items:
        by_bin[magnitude_bin_0p5(mag)].append(eid)
    for b in list(by_bin):
        ids = sorted(set(by_bin[b]))
        rng.shuffle(ids)
        by_bin[b] = ids

    n = sum(len(v) for v in by_bin.values())
    global_target = largest_remainder_three_way(n, ratios, roles)
    bins = sorted(by_bin.keys())
    quota = {b: {r: 0 for r in roles} for b in bins}

    for b in bins:
        m = len(by_bin[b])
        raw = [m * ratio_map[r] for r in roles]
        fl = [int(math.floor(x)) for x in raw]
        for i, r in enumerate(roles):
            quota[b][r] = fl[i]
        rem_b = m - sum(fl)
        order = sorted(range(3), key=lambda i: (-(raw[i] - fl[i]), i))
        for j in range(rem_b):
            quota[b][roles[order[j]]] += 1

    def bounds(b, r):
        ideal = len(by_bin[b]) * ratio_map[r]
        return int(math.floor(ideal)), int(math.ceil(ideal))

    current = {r: sum(quota[b][r] for b in bins) for r in roles}
    for _ in range(10000):
        if all(current[r] == global_target[r] for r in roles):
            break
        over = [r for r in roles if current[r] > global_target[r]]
        under = [r for r in roles if current[r] < global_target[r]]
        moved = False
        for ro in over:
            for ru in under:
                for b in bins:
                    lo_o, hi_o = bounds(b, ro)
                    lo_u, hi_u = bounds(b, ru)
                    if quota[b][ro] > lo_o and quota[b][ru] < hi_u:
                        quota[b][ro] -= 1
                        quota[b][ru] += 1
                        current[ro] -= 1
                        current[ru] += 1
                        moved = True
                        break
                if moved:
                    break
            if moved:
                break
        if not moved:
            for ro in over:
                for ru in under:
                    for b in bins:
                        if quota[b][ro] > 0:
                            quota[b][ro] -= 1
                            quota[b][ru] += 1
                            current[ro] -= 1
                            current[ru] += 1
                            moved = True
                            break
                    if moved:
                        break
                if moved:
                    break
        if not moved:
            raise RuntimeError(
                f'cannot reconcile bin quotas with global targets: {current} vs {global_target}'
            )
    else:
        raise RuntimeError('quota adjustment did not converge')

    out = {r: [] for r in roles}
    for b in bins:
        ids = by_bin[b]
        idx = 0
        for r in roles:
            take = quota[b][r]
            out[r].extend(ids[idx:idx + take])
            idx += take
        if idx != len(ids):
            raise RuntimeError(f'bin {b} assignment incomplete')
    return {r: sorted(out[r]) for r in roles}, global_target, quota

def membership_sha256(splits: dict) -> str:
    payload = {k: list(splits[k]) for k in ('train', 'validation', 'calibration', 'test')}
    return hashlib.sha256(json.dumps(payload, ensure_ascii=False, sort_keys=True).encode('utf-8')).hexdigest()

def formal_policy_dict():
    return {
        'policy': 'preassigned_training_plus_stratified_validation_pool',
        'seed': SEED,
        'validation_pool_ratios': {'validation': 0.50, 'calibration': 0.25, 'test': 0.25},
        'magnitude_bins': '0.5',
        'train_source': 'training',
        'non_train_source': 'validation',
        'combined_inventory_sha256': COMBINED_INVENTORY_SHA256,
        'windows': WINDOWS,
        'label_horizon': LABEL_HORIZON,
        'min_label_valid_fraction': MIN_VALID,
        'min_window_valid_fraction': MIN_VALID,
    }

def build_formal_manifest():
    metadata, event_sources, file_audit = _read_event_metadata(STAGED_ARCHIVE)
    duplicate_ids = {eid: files for eid, files in event_sources.items() if len(files) > 1}
    if duplicate_ids:
        raise RuntimeError(f'duplicate event IDs in staged archive: {len(duplicate_ids)}')

    records, load_stats = load_station_records(
        STAGED_ARCHIVE,
        min_full_valid_fraction=MIN_VALID,
        min_series_length=max(WINDOWS),
        label_horizon=LABEL_HORIZON,
        require_label_horizon=True,
    )
    common_keys = common_cohort_keys(records, WINDOWS, MIN_VALID)
    events = _event_audits(records, metadata, common_keys)
    events = [e for e in events if e.n_common_records > 0]
    allowed = {e.event_id for e in events}
    records = [r for r in records if r.event_id in allowed]
    common_keys = {k for k in common_keys if k.split('\t', 1)[0] in allowed}

    source_by_eid = {}
    for e in events:
        base = Path(e.source_file).name
        role = SOURCE_ROLE_BY_BASENAME.get(base)
        if role is None:
            raise RuntimeError(f'event {e.event_id} source file not in inventory: {e.source_file}')
        source_by_eid[e.event_id] = role

    train_events = [e for e in events if source_by_eid[e.event_id] == 'training']
    val_pool_events = [e for e in events if source_by_eid[e.event_id] == 'validation']

    eligible_basenames = {Path(e.source_file).name for e in events}
    excluded = [row for row in all_inventory if row['basename'] not in eligible_basenames]

    val_items = [(e.event_id, e.magnitude) for e in val_pool_events]
    val_roles, global_target, bin_quota = allocate_validation_pool(val_items, seed=SEED)

    splits = {
        'train': sorted(e.event_id for e in train_events),
        'validation': val_roles['validation'],
        'calibration': val_roles['calibration'],
        'test': val_roles['test'],
    }

    assert set(splits['train']) == {e.event_id for e in train_events}
    non_train = set(splits['validation']) | set(splits['calibration']) | set(splits['test'])
    assert non_train == {e.event_id for e in val_pool_events}
    assert not (set(splits['train']) & non_train)
    for eid in splits['train']:
        assert source_by_eid[eid] == 'training'
    for eid in non_train:
        assert source_by_eid[eid] == 'validation'

    config = {
        'format_version': 3,
        'windows': list(WINDOWS),
        'label_horizon': LABEL_HORIZON,
        'min_label_valid_fraction': MIN_VALID,
        'min_window_valid_fraction': MIN_VALID,
        'cohort': 'common_EW10_to_EW40',
        'split_policy': formal_policy_dict(),
        'split_seed': SEED,
    }
    fingerprint = _fingerprint(events, records, config)
    validation = validate_split(splits, [e.event_id for e in events])
    if not validation['valid']:
        raise RuntimeError(f'formal split validation failed: {validation}')

    cohort_records = [r for r in records if f'{r.event_id}\t{r.station_id}' in common_keys]
    split_check = validate_event_split(
        splits,
        cohort_records,
        required_groups=('train', 'validation', 'calibration', 'test'),
        require_complete_coverage=True,
    )
    if not split_check['valid']:
        raise RuntimeError(f'formal split does not cover common cohort: {split_check}')

    mem_hash = membership_sha256(splits)
    manifest = {
        **config,
        'data_root': str(STAGED_ARCHIVE.resolve()),
        'data_fingerprint_sha256': fingerprint,
        'split_membership_sha256': mem_hash,
        'splits': splits,
        'validation': validation,
        'n_events': len(events),
        'n_station_records': len(records),
        'n_common_cohort_records': len(common_keys),
        'source_counts': {
            'training_files': EXPECTED_TRAIN_COUNT,
            'validation_files': EXPECTED_VALIDATION_COUNT,
            'eligible_train_events': len(splits['train']),
            'eligible_validation_pool_events': len(val_pool_events),
            'excluded_source_files': len(excluded),
        },
        'role_counts': {k: len(v) for k, v in splits.items()},
        'validation_pool_global_targets': global_target,
        'training_inventory_sha256': TRAIN_INVENTORY_SHA256,
        'validation_inventory_sha256': VAL_INVENTORY_SHA256,
        'combined_inventory_sha256': COMBINED_INVENTORY_SHA256,
    }

    PREPARED_DIR.mkdir(parents=True, exist_ok=True)
    (PREPARED_DIR / 'split_manifest.json').write_text(
        json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8'
    )
    (PREPARED_DIR / 'split_distribution.json').write_text(json.dumps({
        'role_counts': manifest['role_counts'],
        'validation_pool_global_targets': global_target,
        'bin_quota': {b: q for b, q in bin_quota.items()},
    }, ensure_ascii=False, indent=2), encoding='utf-8')
    (PREPARED_DIR / 'audit_summary.json').write_text(json.dumps({
        'n_duplicate_event_ids': 0,
        'duplicate_event_ids': {},
        'loader': load_stats,
        'n_events_used': len(events),
        'n_records_used': len(records),
        'n_common_cohort_records': len(common_keys),
        'excluded_source_files': excluded,
        'data_fingerprint_sha256': fingerprint,
        'split_membership_sha256': mem_hash,
        'policy': formal_policy_dict(),
    }, ensure_ascii=False, indent=2), encoding='utf-8')

    split_rows = []
    by_id = {e.event_id: e for e in events}
    for name in ('train', 'validation', 'calibration', 'test'):
        for eid in splits[name]:
            row = asdict(by_id[eid])
            row['split'] = name
            row['source_role'] = source_by_eid[eid]
            row['magnitude_bin_0p5'] = magnitude_bin_0p5(by_id[eid].magnitude)
            split_rows.append(row)
    _write_csv(
        PREPARED_DIR / 'event_split.csv',
        split_rows,
        [
            'split', 'source_role', 'magnitude_bin_0p5', 'event_id', 'source_file',
            'origin_time', 'magnitude', 'n_station_records', 'n_common_records',
            'has_event_positive', 'max_final_class',
        ],
    )
    _write_csv(
        PREPARED_DIR / 'file_audit.csv',
        file_audit,
        ['source_file', 'status', 'error', 'event_id', 'origin_time', 'n_raw_stations'],
    )
    return manifest

def existing_formal_is_compatible(m: dict) -> bool:
    policy = m.get('split_policy') or {}
    try:
        same_root = Path(str(m.get('data_root', ''))).resolve() == STAGED_ARCHIVE.resolve()
    except Exception:
        same_root = str(m.get('data_root', '')) == str(STAGED_ARCHIVE.resolve())
    return (
        same_root
        and list(m.get('windows', [])) == WINDOWS
        and int(m.get('label_horizon', -1)) == LABEL_HORIZON
        and policy.get('policy') == 'preassigned_training_plus_stratified_validation_pool'
        and int(policy.get('seed', -1)) == SEED
        and policy.get('combined_inventory_sha256') == COMBINED_INVENTORY_SHA256
        and m.get('combined_inventory_sha256') == COMBINED_INVENTORY_SHA256
    )

if REBUILD_SPLIT:
    raise RuntimeError(
        'REBUILD_SPLIT=True 已被停用（防止覆寫凍結 split）。'
        f'若資料或政策變更，請改 PREPARED_VERSION（目前 {PREPARED_VERSION}）並使用新目錄。'
    )

if SPLIT_MANIFEST.is_file():
    manifest = json.loads(SPLIT_MANIFEST.read_text(encoding='utf-8'))
    if not existing_formal_is_compatible(manifest):
        raise RuntimeError(
            '既有 split_manifest.json 與目前 inventory/policy 不符；'
            f'請改 PREPARED_VERSION（目前 {PREPARED_VERSION}）後重建，勿覆寫舊正式 split。'
        )
    print('Loaded frozen formal split:', SPLIT_MANIFEST)
else:
    assert CREATE_SPLIT_IF_MISSING
    manifest = build_formal_manifest()
    print('Created formal preassigned split:', SPLIT_MANIFEST)

assert manifest['validation']['valid']
assert existing_formal_is_compatible(manifest)
counts = {k: len(v) for k, v in manifest['splits'].items()}
print('Fingerprint:', manifest['data_fingerprint_sha256'])
print('Membership:', manifest.get('split_membership_sha256'))
display(pd.DataFrame([{'split': k, 'events': v} for k, v in counts.items()]))

audit = json.loads((PREPARED_DIR / 'audit_summary.json').read_text(encoding='utf-8'))
assert audit['n_duplicate_event_ids'] == 0
split_df = pd.read_csv(PREPARED_DIR / 'event_split.csv')
display(
    split_df.groupby(['split', 'source_role']).agg(
        events=('event_id', 'nunique'),
        records=('n_station_records', 'sum'),
        positive_rate=('has_event_positive', 'mean'),
        median_mag=('magnitude', 'median'),
    ).reset_index()
)

# Quick diagnostic split: training-only random four-way via prepare_ssif_dataset
if not QUICK_SPLIT_MANIFEST.is_file():
    QUICK_DIAG_DIR.mkdir(parents=True, exist_ok=True)
    run_checked([
        'python', 'prepare_ssif_dataset.py', 'audit-split',
        '--data-dir', str(TRAIN_DATA), '--output-dir', str(QUICK_DIAG_DIR),
        '--windows', *map(str, WINDOWS), '--label-horizon', str(LABEL_HORIZON),
        '--min-label-valid-fraction', str(MIN_VALID),
        '--min-window-valid-fraction', str(MIN_VALID),
        '--train-ratio', '0.70', '--validation-ratio', '0.10',
        '--calibration-ratio', '0.10', '--test-ratio', '0.10',
        '--split-candidates', '5000', '--seed', str(SEED),
    ], cwd=REPO_ROOT)
quick_manifest = json.loads(QUICK_SPLIT_MANIFEST.read_text(encoding='utf-8'))
assert quick_manifest['validation']['valid']
formal_non_train = (
    set(manifest['splits']['validation'])
    | set(manifest['splits']['calibration'])
    | set(manifest['splits']['test'])
)
quick_all = set().union(*(set(v) for v in quick_manifest['splits'].values()))
assert not (quick_all & formal_non_train), 'quick diagnostic split leaked formal validation-pool events'
print('Quick diagnostic split ready:', QUICK_SPLIT_MANIFEST)

## 6. 訓練命令

- Quick：`TRAIN_DATA` + `quick_diagnostic/split_manifest.json`（僅診斷）
- Formal：`STAGED_ARCHIVE` + 凍結 formal `split_manifest.json`

In [ ]:
def train_command(output_dir, windows, epochs, *, data_dir, split_manifest):
    command = [
        'python', 'train_ssif_v3.py', 'train-all',
        '--data-dir', str(data_dir),
        '--split-manifest', str(split_manifest),
        '--output-dir', str(output_dir),
        '--windows', *map(str, windows),
        '--label-horizon', str(LABEL_HORIZON),
        '--cohort', 'common',
        '--epochs', str(epochs),
        '--batch-size', str(BATCH_SIZE),
        '--eval-batch-size', str(EVAL_BATCH_SIZE),
        '--lr', '3e-4',
        '--weight-decay', '1e-2',
        '--warmup-ratio', '0.10',
        '--min-precision', str(MIN_PRECISION),
        '--hidden-size', '192',
        '--num-layers', '4',
        '--num-heads', '4',
        '--ff-mult', '2',
        '--dropout', '0.1',
        '--conv1', '96',
        '--conv2', '192',
        '--loss-cls', '0.45',
        '--loss-alert', '0.35',
        '--loss-ordinal', '0.15',
        '--loss-consistency', '0.05',
        '--seed', str(SEED),
        '--window-seed-mode', 'same',
        '--patience', '6',
        '--workers', str(WORKERS),
        '--parallel-windows', str(PARALLEL_EW_JOBS),
    ]
    if torch.cuda.is_available():
        command.append('--amp')
    return command

def quick_train_command(output_dir, windows, epochs):
    return train_command(
        output_dir, windows, epochs,
        data_dir=TRAIN_DATA,
        split_manifest=QUICK_SPLIT_MANIFEST,
    )

def formal_train_command(output_dir, windows, epochs):
    return train_command(
        output_dir, windows, epochs,
        data_dir=STAGED_ARCHIVE,
        split_manifest=SPLIT_MANIFEST,
    )


## 7. EW10 一個 epoch 快速測試（training-only diagnostic）

此節指標僅供診斷，**不可**與 formal locked-test 比較。

In [ ]:
if RUN_QUICK_TRAIN:
    if QUICK_MODEL_DIR.exists() and any(QUICK_MODEL_DIR.iterdir()):
        assert OVERWRITE_QUICK_MODEL
        shutil.rmtree(QUICK_MODEL_DIR)
    run_checked(quick_train_command(QUICK_MODEL_DIR, [10], 1), cwd=REPO_ROOT)
    assert (QUICK_MODEL_DIR / 'EW10' / 'best.pt').is_file()
    quick = json.loads((QUICK_MODEL_DIR / 'summary.json').read_text(encoding='utf-8'))[0]
    display(pd.DataFrame([{
        'window': quick['window'],
        'best_epoch': quick['best_epoch'],
        'threshold': quick['threshold'],
        'precision': quick['test']['alert']['precision'],
        'pod': quick['test']['alert']['pod'],
        'f1': quick['test']['alert']['f1'],
        'fpr': quick['test']['alert']['fpr'],
        'note': 'diagnostic(training-only)',
    }]))
    print('PASS: EW10 quick diagnostic training')
else:
    print('RUN_QUICK_TRAIN=False')

## 8. 正式訓練 EW10–EW40

執行前請先確認第 7 節 quick diagnostic 已通過，並在第 2 節設定 `RUN_FULL_TRAIN=True`。正式訓練使用 **staged archive + frozen preassigned manifest**。

本節會以 `--parallel-windows=PARALLEL_EW_JOBS` **並行訓練多個 EW 視窗**（預設 2，適合單顆 A100）。資料只載入一次，再以多執行緒同時訓練不同 EW，以縮短總壁鐘時間。

**Colab 執行階段建議（A100）**
- 硬體加速器：A100 GPU
- **大量 RAM / High-RAM：建議開啟**（約 50 萬筆 station records + 2 個並行訓練迴圈會吃系統 RAM；High-RAM 與 GPU 顯存是分開的）
- 若出現 CUDA OOM：把第 2 節 `PARALLEL_EW_JOBS` 改為 `1`，或把 `BATCH_SIZE` 降到 `8`

步驟：
1. 前置檢查：資料、固定 split、EW10–EW40 視窗、並行度與運算裝置。
2. 輸出目錄：防止不小心覆蓋既有正式模型。
3. 訓練計畫：列出 epoch、batch、seed、`PARALLEL_EW_JOBS` 與輸出位置。
4. 即時訓練：多個 EW 的 epoch 日誌可能交錯出現；進度表會分別更新。
5. 產物驗證：逐一檢查每個 EW 的 checkpoint、history 與 metrics。
6. 完成摘要：列出最佳 epoch 與 locked-test 指標。

即時終端輸出會保存至 `full_training_live.log`；可機器讀取的目前狀態會保存至 `full_training_progress.json`。


In [ ]:
import re
import time
from collections import deque

FULL_EPOCHS = 30
PROGRESS_PATH = REPORT_DIR / 'full_training_progress.json'
LIVE_LOG_PATH = REPORT_DIR / 'full_training_live.log'
SUBPROCESS_LOG_TAIL_LINES = 80
latest_subprocess_lines = deque(maxlen=SUBPROCESS_LOG_TAIL_LINES)
current_stage = '尚未開始'

_EPOCH_PATTERN = re.compile(
    r'\[EW(\d+)\] epoch\s+(\d+)/(\d+)\s+'
    r'loss=(\S+) val_AP=(\S+) thr=(\S+) '
    r'P=(\S+) POD=(\S+) F1=(\S+)'
)
_EARLY_STOP_PATTERN = re.compile(r'\[EW(\d+)\] early stopping at epoch (\d+)')
_START_PATTERN = re.compile(r'\[EW(\d+)\] start training')
_FINISHED_PATTERN = re.compile(
    r'\[EW(\d+)\] finished best_epoch=(\S+) thr=(\S+) test_F1=(\S+)'
)

def _seconds_text(seconds):
    seconds = int(max(0, seconds))
    hours, remainder = divmod(seconds, 3600)
    minutes, seconds = divmod(remainder, 60)
    return f'{hours:02d}:{minutes:02d}:{seconds:02d}'

def _new_progress_rows():
    return {
        w: {
            'window': f'EW{w:02d}',
            'status': '等待',
            'epoch': 0,
            'max_epoch': FULL_EPOCHS,
            'progress': '0%',
            'loss': None,
            'val_AP': None,
            'threshold': None,
            'precision': None,
            'POD': None,
            'F1': None,
            'elapsed': '00:00:00',
        }
        for w in WINDOWS
    }

full_progress = _new_progress_rows()
_window_started_at = {}

def _progress_frame():
    columns = [
        'window', 'status', 'epoch', 'max_epoch', 'progress', 'loss',
        'val_AP', 'threshold', 'precision', 'POD', 'F1', 'elapsed',
    ]
    return pd.DataFrame([full_progress[w] for w in WINDOWS])[columns]

def _save_progress(overall_status, message):
    payload = {
        'updated_at': datetime.now().astimezone().isoformat(timespec='seconds'),
        'overall_status': overall_status,
        'message': message,
        'current_stage': current_stage,
        'parallel_ew_jobs': PARALLEL_EW_JOBS,
        'last_log_lines': list(latest_subprocess_lines),
        'model_dir': str(FULL_MODEL_DIR),
        'live_log': str(LIVE_LOG_PATH),
        'windows': [full_progress[w] for w in WINDOWS],
    }
    PROGRESS_PATH.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2),
        encoding='utf-8',
    )

def _update_progress_display(handle):
    frame = _progress_frame()
    if handle is None:
        display(frame)
    else:
        handle.update(frame)

if RUN_FULL_TRAIN:
    overall_started_at = time.time()
    current_stage = '步驟 1/6：前置檢查'
    print('[步驟 1/6] 前置檢查')
    assert STAGED_ARCHIVE.is_dir(), f'staged archive 不存在：{STAGED_ARCHIVE}'
    assert SPLIT_MANIFEST.is_file(), f'找不到固定 split：{SPLIT_MANIFEST}'
    assert WINDOWS == [10, 15, 20, 25, 30, 35, 40], f'正式視窗設定錯誤：{WINDOWS}'
    assert PARALLEL_EW_JOBS >= 1, f'PARALLEL_EW_JOBS must be >= 1, got {PARALLEL_EW_JOBS}'
    device_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
    print(f'  ✓ staged JSON：{EXPECTED_STAGED_COUNT:,}')
    print(f'  ✓ 固定 split：{SPLIT_MANIFEST}')
    print(f'  ✓ 視窗：{WINDOWS}')
    print(f'  ✓ 並行 EW 數：{PARALLEL_EW_JOBS}')
    print(f'  ✓ 運算裝置：{device_name}')
    if PARALLEL_EW_JOBS > 1:
        print('  ✓ Colab 建議：開啟「大量 RAM / High-RAM」（系統 RAM，不是 GPU VRAM）')
    if not (QUICK_MODEL_DIR / 'EW10' / 'best.pt').is_file():
        print('  ⚠ 未偵測到 EW10 quick checkpoint；請確認第 7 節是否已通過。')

    current_stage = '步驟 2/6：準備正式模型輸出目錄'
    print('\n[步驟 2/6] 準備正式模型輸出目錄')
    if FULL_MODEL_DIR.exists() and any(FULL_MODEL_DIR.iterdir()):
        assert OVERWRITE_FULL_MODEL, (
            f'正式模型已存在：{FULL_MODEL_DIR}。'
            '請改 run 名稱，或確認後設定 OVERWRITE_FULL_MODEL=True。'
        )
        print(f'  ! 已明確允許覆蓋，移除舊目錄：{FULL_MODEL_DIR}')
        shutil.rmtree(FULL_MODEL_DIR)
    FULL_MODEL_DIR.mkdir(parents=True, exist_ok=True)
    print(f'  ✓ 輸出目錄已就緒：{FULL_MODEL_DIR}')

    current_stage = '步驟 3/6：確認訓練計畫'
    print('\n[步驟 3/6] 確認訓練計畫')
    training_plan = pd.DataFrame([{
        'windows': ', '.join(f'EW{w:02d}' for w in WINDOWS),
        'epochs_per_window': FULL_EPOCHS,
        'batch_size': BATCH_SIZE,
        'eval_batch_size': EVAL_BATCH_SIZE,
        'seed': SEED,
        'min_precision': MIN_PRECISION,
        'workers': WORKERS,
        'parallel_ew_jobs': PARALLEL_EW_JOBS,
        'amp': bool(torch.cuda.is_available()),
        'split_policy': 'preassigned',
        'data_dir': str(STAGED_ARCHIVE),
        'high_ram_recommended': PARALLEL_EW_JOBS > 1,
    }])
    display(training_plan)
    command = formal_train_command(FULL_MODEL_DIR, WINDOWS, FULL_EPOCHS)
    print('  ✓ validation 選最佳 epoch；calibration 選 threshold；test 保持鎖定到最後評估。')
    print(f'  ✓ 即時紀錄：{LIVE_LOG_PATH}')
    print(f'  ✓ 狀態檔：{PROGRESS_PATH}')
    print('  ✓ 執行命令：' + ' '.join(map(str, command)))

    current_stage = '步驟 4/6：載入資料、驗證 split 並建立 common cohort'
    print('\n[步驟 4/6] 開始正式訓練（可並行多個 EW）；下表會逐 epoch 更新')
    for w in WINDOWS:
        full_progress[w]['status'] = '排隊中'
    _save_progress(
        'starting',
        f'正在載入資料並以 PARALLEL_EW_JOBS={PARALLEL_EW_JOBS} 訓練 EW10–EW40。',
    )
    progress_display = display(_progress_frame(), display_id=True)
    process = None
    try:
        process_env = os.environ.copy()
        process_env['PYTHONUNBUFFERED'] = '1'
        process = subprocess.Popen(
            command,
            cwd=str(REPO_ROOT),
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            env=process_env,
        )
        with LIVE_LOG_PATH.open('w', encoding='utf-8') as live_log:
            assert process.stdout is not None
            for line in process.stdout:
                latest_subprocess_lines.append(line.rstrip('\n'))
                print(line, end='')
                live_log.write(line)
                live_log.flush()

                start_match = _START_PATTERN.search(line)
                if start_match:
                    w = int(start_match.group(1))
                    _window_started_at.setdefault(w, time.time())
                    full_progress[w]['status'] = '訓練中'
                    current_stage = f'並行訓練中（含 EW{w:02d}）'
                    _save_progress('running', f'EW{w:02d} 開始訓練')
                    _update_progress_display(progress_display)
                    continue

                epoch_match = _EPOCH_PATTERN.search(line)
                if epoch_match:
                    w = int(epoch_match.group(1))
                    epoch = int(epoch_match.group(2))
                    max_epoch = int(epoch_match.group(3))
                    current_stage = f'並行訓練中（EW{w:02d} epoch {epoch}/{max_epoch}）'
                    _window_started_at.setdefault(w, time.time())
                    row = full_progress[w]
                    row.update({
                        'status': '訓練中',
                        'epoch': epoch,
                        'max_epoch': max_epoch,
                        'progress': f'{epoch / max_epoch:.0%}',
                        'loss': float(epoch_match.group(4)),
                        'val_AP': float(epoch_match.group(5)),
                        'threshold': float(epoch_match.group(6)),
                        'precision': float(epoch_match.group(7)),
                        'POD': float(epoch_match.group(8)),
                        'F1': float(epoch_match.group(9)),
                        'elapsed': _seconds_text(time.time() - _window_started_at[w]),
                    })
                    running = sum(1 for item in full_progress.values() if item['status'] == '訓練中')
                    _save_progress(
                        'running',
                        f'EW{w:02d} epoch {epoch}/{max_epoch}；目前並行中約 {running} 個視窗',
                    )
                    _update_progress_display(progress_display)
                    continue

                early_stop_match = _EARLY_STOP_PATTERN.search(line)
                if early_stop_match:
                    w = int(early_stop_match.group(1))
                    full_progress[w]['status'] = '提前停止；校準與測試中'
                    _save_progress(
                        'running',
                        f'EW{w:02d} 於 epoch {early_stop_match.group(2)} 提前停止；正在完成後處理。',
                    )
                    _update_progress_display(progress_display)
                    continue

                finished_match = _FINISHED_PATTERN.search(line)
                if finished_match:
                    w = int(finished_match.group(1))
                    full_progress[w]['status'] = '完成；產物待總驗證'
                    full_progress[w]['progress'] = '100%'
                    full_progress[w]['threshold'] = float(finished_match.group(3))
                    full_progress[w]['F1'] = float(finished_match.group(4))
                    if w in _window_started_at:
                        full_progress[w]['elapsed'] = _seconds_text(
                            time.time() - _window_started_at[w]
                        )
                    _save_progress('running', f'EW{w:02d} 完成')
                    _update_progress_display(progress_display)

        returncode = process.wait()
        if returncode:
            raise RuntimeError(
                f'正式訓練程序失敗（exit code={returncode}）。'
                f' 子程序最後 {len(latest_subprocess_lines)} 行已寫入狀態檔；'
                f'完整內容請查看 {LIVE_LOG_PATH}'
            )
    except BaseException as exc:
        if process is not None and process.poll() is None:
            process.terminate()
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill()
        for row in full_progress.values():
            if row['status'] in {'訓練中', '排隊中', '提前停止；校準與測試中'}:
                row['status'] = '使用者中止' if isinstance(exc, KeyboardInterrupt) else '失敗'
        failure_status = 'interrupted' if isinstance(exc, KeyboardInterrupt) else 'failed'
        current_stage = f'{current_stage}（失敗）'
        failure_message = f'{type(exc).__name__}: {exc}'
        _save_progress(failure_status, failure_message)
        _update_progress_display(progress_display)
        print(f'  ✗ {failure_message}')
        if latest_subprocess_lines:
            print(f'\n[子程序最後 {len(latest_subprocess_lines)} 行]')
            print('\n'.join(latest_subprocess_lines))
        raise

    current_stage = '步驟 5/6：逐一驗證訓練產物'
    print('\n[步驟 5/6] 逐一驗證 EW10–EW40 訓練產物')
    required_artifacts = ('best.pt', 'history.json', 'metrics.json')
    missing_artifacts = []
    for w in WINDOWS:
        run_dir = FULL_MODEL_DIR / f'EW{w:02d}'
        missing = [name for name in required_artifacts if not (run_dir / name).is_file()]
        if missing:
            full_progress[w]['status'] = '產物不完整'
            missing_artifacts.append(f'EW{w:02d}: {missing}')
            print(f'  ✗ EW{w:02d} 缺少：{", ".join(missing)}')
        else:
            full_progress[w]['status'] = '完成'
            full_progress[w]['progress'] = '100%'
            print(f'  ✓ EW{w:02d}: best.pt、history.json、metrics.json')
    summary_path = FULL_MODEL_DIR / 'summary.json'
    if not summary_path.is_file():
        missing_artifacts.append('缺少 summary.json')
        print('  ✗ 缺少 summary.json')
    _update_progress_display(progress_display)
    if missing_artifacts:
        message = '；'.join(missing_artifacts)
        _save_progress('failed', message)
        raise AssertionError(message)

    current_stage = '步驟 6/6：建立正式訓練摘要'
    print('\n[步驟 6/6] 正式訓練完成摘要')
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    summary_rows = []
    for item in summary:
        alert = item['test']['alert']
        summary_rows.append({
            'window': f"EW{int(item['window']):02d}",
            'best_epoch': item['best_epoch'],
            'threshold': item['threshold'],
            'precision': alert['precision'],
            'POD': alert['pod'],
            'F1': alert['f1'],
            'FPR': alert['fpr'],
            'test_n': item['test']['n_samples'],
        })
    summary_table = pd.DataFrame(summary_rows).sort_values('window')
    display(summary_table)
    total_elapsed = _seconds_text(time.time() - overall_started_at)
    completion_message = (
        f'PASS: EW10–EW40 共 {len(WINDOWS)} 個模型完成；'
        f'PARALLEL_EW_JOBS={PARALLEL_EW_JOBS}；總耗時 {total_elapsed}。'
    )
    current_stage = '已完成'
    _save_progress('completed', completion_message)
    _update_progress_display(progress_display)
    print(completion_message)
    print(f'完整終端紀錄：{LIVE_LOG_PATH}')
    print(f'最終狀態紀錄：{PROGRESS_PATH}')
else:
    print('RUN_FULL_TRAIN=False；第 7 節 quick diagnostic 通過後，請在第 2 節改為 True 再執行本節。')


## 9. 彙整正式模型與繪圖

In [ ]:
def result_table(model_dir):
    path = model_dir / 'summary.json'
    if not path.is_file():
        return pd.DataFrame()
    rows = []
    for x in json.loads(path.read_text(encoding='utf-8')):
        a = x['test']['alert']
        rows.append({
            'window': x['window'],
            'best_epoch': x['best_epoch'],
            'threshold': x['threshold'],
            'precision': a['precision'],
            'pod': a['pod'],
            'f1': a['f1'],
            'fpr': a['fpr'],
            'n': x['test']['n_samples'],
        })
    return pd.DataFrame(rows).sort_values('window')

results = result_table(FULL_MODEL_DIR)
if results.empty:
    print('尚無正式 summary.json')
else:
    display(results)
    results.to_csv(REPORT_DIR / 'model_performance_by_window.csv', index=False, encoding='utf-8-sig')
    plt.figure(figsize=(9, 5))
    for col in ['precision', 'pod', 'f1']:
        plt.plot(results['window'], results[col], marker='o', label=col.upper())
    plt.xlabel('Early window (s)')
    plt.ylabel('Score')
    plt.ylim(0, 1.02)
    plt.xticks(WINDOWS)
    plt.grid(alpha=.3)
    plt.legend()
    plt.tight_layout()
    plt.savefig(REPORT_DIR / 'test_metrics_by_window.png', dpi=180)
    plt.show()

## 10. Checkpoint 與資料指紋稽核

In [ ]:
rows = []
for w in WINDOWS:
    path = FULL_MODEL_DIR / f'EW{w:02d}' / 'best.pt'
    if not path.is_file():
        rows.append({'window': w, 'exists': False})
        continue
    payload = torch.load(path, map_location='cpu', weights_only=False)
    meta = payload.get('training_metadata', {})
    rows.append({
        'window': w,
        'exists': True,
        'checkpoint_window': payload.get('window'),
        'best_epoch': meta.get('best_epoch'),
        'threshold': payload.get('alert_probability_threshold'),
        'fingerprint_matches': meta.get('data_fingerprint_sha256') == manifest['data_fingerprint_sha256'],
        'label_horizon': meta.get('label_horizon'),
        'cohort': meta.get('cohort'),
    })
checkpoint_audit = pd.DataFrame(rows)
display(checkpoint_audit)
if RUN_FULL_TRAIN:
    assert checkpoint_audit['exists'].all()
    assert checkpoint_audit['fingerprint_matches'].all()
    print('PASS: checkpoints match frozen data fingerprint')

## 11. 選擇性：完全獨立 archive inference

In [ ]:
if RUN_EXTERNAL_EVALUATION:
    assert EXTERNAL_DATA.is_dir() and any(EXTERNAL_DATA.rglob('*.json')), (
        f'找不到獨立 evaluation archive（含子目錄 *.json）：{EXTERNAL_DATA}'
    )
    assert (FULL_MODEL_DIR / 'summary.json').is_file()
    if EXTERNAL_OUTPUT_DIR.exists() and any(EXTERNAL_OUTPUT_DIR.iterdir()):
        raise RuntimeError('外部評估輸出已存在；請使用新的輸出目錄')
    run_checked([
        'python', 'train_ssif_v3.py', 'evaluate-all',
        '--data-dir', str(EXTERNAL_DATA),
        '--model-root', str(FULL_MODEL_DIR),
        '--output-dir', str(EXTERNAL_OUTPUT_DIR),
        '--windows', *map(str, WINDOWS),
        '--label-horizon', str(LABEL_HORIZON),
        '--cohort', 'common',
        '--batch-size', '128',
        '--workers', str(WORKERS),
    ], cwd=REPO_ROOT)
else:
    print('RUN_EXTERNAL_EVALUATION=False')

## 12. 保存 run inventory

In [ ]:
inventory = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(timespec='seconds').replace('+00:00', 'Z'),
    'repository_commit': REPO_SHA,
    'drive_data_url': DRIVE_DATA_URL,
    'dataset_root': str(DATASET_ROOT.resolve()),
    'train_data': str(TRAIN_DATA.resolve()),
    'validation_data': str(VALIDATION_DATA.resolve()),
    'staged_archive': str(STAGED_ARCHIVE.resolve()),
    'training_inventory_sha256': TRAIN_INVENTORY_SHA256,
    'validation_inventory_sha256': VAL_INVENTORY_SHA256,
    'combined_inventory_sha256': COMBINED_INVENTORY_SHA256,
    'data_fingerprint_sha256': manifest['data_fingerprint_sha256'],
    'split_membership_sha256': manifest.get('split_membership_sha256'),
    'split_manifest': str(SPLIT_MANIFEST),
    'quick_split_manifest': str(QUICK_SPLIT_MANIFEST),
    'prepared_version': PREPARED_VERSION,
    'split_policy': manifest.get('split_policy'),
    'role_counts': {k: len(v) for k, v in manifest['splits'].items()},
    'source_counts': manifest.get('source_counts'),
    'quick_model_dir': str(QUICK_MODEL_DIR),
    'full_model_dir': str(FULL_MODEL_DIR),
    'windows': WINDOWS,
    'seed': SEED,
    'label_horizon': LABEL_HORIZON,
    'min_valid_fraction': MIN_VALID,
    'min_precision': MIN_PRECISION,
    'flags': {
        'quick': RUN_QUICK_TRAIN,
        'full': RUN_FULL_TRAIN,
        'external': RUN_EXTERNAL_EVALUATION,
    },
}
(REPORT_DIR / 'run_inventory.json').write_text(
    json.dumps(inventory, ensure_ascii=False, indent=2), encoding='utf-8'
)
print(json.dumps(inventory, ensure_ascii=False, indent=2))

## 執行順序

1. 先確認 `/content/drive/MyDrive/00_SSIF/Data_formodel` 存在，再執行到第 7 節（資料、preassigned split、EW10 quick diagnostic）。
2. 保持同一份 formal `split_manifest.json`，不要依模型結果重切資料。
3. 將 `RUN_FULL_TRAIN=True` 後執行第 8 節。
4. 執行第 9–10 節，保存表格、圖與 checkpoint 指紋稽核。
5. 只有具備完全獨立事件 archive 時才執行第 11 節。